# Rate-Encoded Spiking LeNet-5 on MNIST

Pure PyTorch spiking LeNet-5 trained with surrogate-gradient BPTT.

| Component | Choice |
|---|---|
| Input encoding | Rate (Bernoulli) coding: pixel intensity in [0,1] = spike probability per time step |
| Neuron | Leaky Integrate-and-Fire (LIF), soft reset |
| Architecture | conv(6,5×5) → LIF → maxpool → conv(16,5×5) → LIF → maxpool → FC120 → LIF → FC84 → LIF → FC10 → LIF |
| Readout | Output spike count over T steps (rate decoding) |
| Loss | Cross-entropy on spike counts |
| Training | BPTT, fast-sigmoid surrogate gradient, Adam + cosine LR |

LIF dynamics per time step $t$:
$$U[t] = \beta\,U[t-1] + W X[t], \qquad S[t] = \Theta(U[t]-\vartheta), \qquad U[t] \leftarrow U[t] - S[t]\,\vartheta$$

Surrogate gradient: $\partial S/\partial U \approx 1/(k|U-\vartheta|+1)^2$.

## 1. Setup

In [ ]:
# !pip install torch torchvision matplotlib scikit-learn tqdm   # uncomment if needed
import time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.auto import tqdm

%matplotlib inline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## 2. Configuration

In [ ]:
CFG = dict(
    T          = 25,      # simulation time steps
    gain       = 1.0,     # rate-encoding gain (spike prob = gain * pixel)
    beta       = 0.9,     # membrane leak factor
    threshold  = 1.0,     # firing threshold
    reset      = "soft",  # "soft" (subtract) or "hard" (zero)
    slope      = 25.0,    # surrogate gradient steepness
    epochs     = 10,
    batch_size = 128,
    lr         = 1e-3,
    seed       = 42,
    data_root  = "./data",
    ckpt       = "best_spiking_lenet_mnist.pt",
)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(CFG["seed"])
CFG

## 3. MNIST data
No normalisation: pixels stay in [0,1] so they can be used directly as spike probabilities.

In [ ]:
tf = transforms.ToTensor()
train_ds = datasets.MNIST(CFG["data_root"], train=True,  download=True, transform=tf)
test_ds  = datasets.MNIST(CFG["data_root"], train=False, download=True, transform=tf)

pin = torch.cuda.is_available()
train_dl = DataLoader(train_ds, CFG["batch_size"], shuffle=True,  num_workers=0, pin_memory=pin, drop_last=True)
test_dl  = DataLoader(test_ds,  CFG["batch_size"], shuffle=False, num_workers=0, pin_memory=pin)
print(f"train: {len(train_ds)}  test: {len(test_ds)}  batches/epoch: {len(train_dl)}")

## 4. Rate encoder

In [ ]:
def rate_encode(x, T, gain=1.0):
    """x: (B,C,H,W) in [0,1]  ->  spikes: (T,B,C,H,W), Bernoulli(p = gain*x) each step."""
    p = (x * gain).clamp(0.0, 1.0)
    return torch.bernoulli(p.unsqueeze(0).expand(T, *p.shape))

In [ ]:
# Visualise encoding of one digit
img, lbl = test_ds[0]
spk = rate_encode(img.unsqueeze(0), CFG["T"], CFG["gain"])[:, 0, 0]   # (T,28,28)

fig, ax = plt.subplots(1, 6, figsize=(14, 2.6))
ax[0].imshow(img[0], cmap="gray"); ax[0].set_title(f"input (label {lbl})")
for i, t in enumerate([0, 1, 2, 3]):
    ax[i+1].imshow(spk[t], cmap="gray"); ax[i+1].set_title(f"spikes t={t}")
ax[5].imshow(spk.mean(0), cmap="gray"); ax[5].set_title(f"mean rate over T={CFG['T']}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 5. Surrogate spike function and LIF neuron

In [ ]:
class FastSigmoidSpike(torch.autograd.Function):
    """Heaviside forward; fast-sigmoid derivative backward."""
    slope = CFG["slope"]

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x > 0).float()

    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        return grad_out / (FastSigmoidSpike.slope * x.abs() + 1.0) ** 2

spike_fn = FastSigmoidSpike.apply


class LIF(nn.Module):
    def __init__(self, beta=0.9, threshold=1.0, reset="soft"):
        super().__init__()
        self.beta, self.threshold, self.reset = beta, threshold, reset
        self.mem = None

    def reset_state(self):
        self.mem = None

    def forward(self, x):
        if self.mem is None or self.mem.shape != x.shape:
            self.mem = torch.zeros_like(x)
        self.mem = self.beta * self.mem + x
        spk = spike_fn(self.mem - self.threshold)
        if self.reset == "soft":
            self.mem = self.mem - spk * self.threshold
        else:
            self.mem = self.mem * (1.0 - spk.detach())
        return spk

## 6. Spiking LeNet-5

In [ ]:
class SpikingLeNet5(nn.Module):
    """
    1x28x28 -> conv5x5(6,pad2) -> LIF -> maxpool2 :  6x14x14
            -> conv5x5(16)     -> LIF -> maxpool2 : 16x5x5
            -> FC 400->120 -> LIF -> FC 120->84 -> LIF -> FC 84->10 -> LIF
    """
    LAYERS = ["conv1", "conv2", "fc1", "fc2", "out"]

    def __init__(self, num_classes=10, beta=0.9, threshold=1.0, reset="soft"):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1   = nn.Linear(16 * 5 * 5, 120)
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, num_classes)
        kw = dict(beta=beta, threshold=threshold, reset=reset)
        self.lifs = nn.ModuleList([LIF(**kw) for _ in range(5)])

    def reset_state(self):
        for l in self.lifs: l.reset_state()

    def step(self, x):
        l1, l2, l3, l4, lo = self.lifs
        s1 = F.max_pool2d(l1(self.conv1(x)), 2)
        s2 = F.max_pool2d(l2(self.conv2(s1)), 2)
        s3 = l3(self.fc1(s2.flatten(1)))
        s4 = l4(self.fc2(s3))
        so = lo(self.fc3(s4))
        return so, (s1, s2, s3, s4, so)

    def forward(self, spikes_in, record=False):
        """spikes_in: (T,B,1,28,28). Returns spike counts (B,10); with record=True also
        per-layer mean firing rates and the output spike train (T,B,10)."""
        self.reset_state()
        T = spikes_in.shape[0]
        counts, rates, out_train = 0.0, [0.0] * 5, []
        for t in range(T):
            so, layers = self.step(spikes_in[t])
            counts = counts + so
            if record:
                out_train.append(so.detach())
                for i, s in enumerate(layers):
                    rates[i] += s.detach().mean().item() / T
        if record:
            return counts, rates, torch.stack(out_train)
        return counts


model = SpikingLeNet5(beta=CFG["beta"], threshold=CFG["threshold"], reset=CFG["reset"]).to(device)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 7. Training and evaluation loops

In [ ]:
def train_epoch(model, loader, opt):
    model.train()
    tot_loss, correct, n = 0.0, 0, 0
    for x, y in tqdm(loader, leave=False, desc="train"):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        counts = model(rate_encode(x, CFG["T"], CFG["gain"]))
        loss = F.cross_entropy(counts, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        tot_loss += loss.item() * y.size(0)
        correct  += (counts.argmax(1) == y).sum().item()
        n        += y.size(0)
    return tot_loss / n, 100.0 * correct / n


@torch.no_grad()
def evaluate(model, loader, return_preds=False):
    model.eval()
    tot_loss, correct, n = 0.0, 0, 0
    rate_sum = np.zeros(5)
    preds, targets = [], []
    for x, y in tqdm(loader, leave=False, desc="eval"):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        counts, rates, _ = model(rate_encode(x, CFG["T"], CFG["gain"]), record=True)
        tot_loss += F.cross_entropy(counts, y).item() * y.size(0)
        p = counts.argmax(1)
        correct  += (p == y).sum().item()
        n        += y.size(0)
        rate_sum += np.array(rates)
        if return_preds:
            preds.append(p.cpu()); targets.append(y.cpu())
    out = dict(loss=tot_loss / n, acc=100.0 * correct / n, rates=rate_sum / len(loader))
    if return_preds:
        out["preds"], out["targets"] = torch.cat(preds).numpy(), torch.cat(targets).numpy()
    return out

## 8. Train

In [ ]:
opt   = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"])
hist  = dict(train_loss=[], train_acc=[], test_loss=[], test_acc=[], rates=[])
best  = 0.0

for ep in range(1, CFG["epochs"] + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_dl, opt)
    te = evaluate(model, test_dl)
    sched.step()

    hist["train_loss"].append(tr_loss); hist["train_acc"].append(tr_acc)
    hist["test_loss"].append(te["loss"]); hist["test_acc"].append(te["acc"])
    hist["rates"].append(te["rates"])
    if te["acc"] > best:
        best = te["acc"]
        torch.save(model.state_dict(), CFG["ckpt"])

    print(f"Ep {ep:02d} | train {tr_loss:.4f} / {tr_acc:.2f}% | test {te['loss']:.4f} / {te['acc']:.2f}% "
          f"| best {best:.2f}% | rates [{', '.join(f'{r:.3f}' for r in te['rates'])}] | {time.time()-t0:.1f}s")

print(f"\nBest test accuracy: {best:.2f}%  -> {CFG['ckpt']}")

## 9. Learning curves and firing rates

In [ ]:
ep = np.arange(1, len(hist["train_loss"]) + 1)
rates = np.array(hist["rates"])

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(ep, hist["train_loss"], "o-", label="train"); ax[0].plot(ep, hist["test_loss"], "s-", label="test")
ax[0].set(title="Loss", xlabel="epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, hist["train_acc"], "o-", label="train"); ax[1].plot(ep, hist["test_acc"], "s-", label="test")
ax[1].set(title="Accuracy (%)", xlabel="epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
for i, name in enumerate(SpikingLeNet5.LAYERS):
    ax[2].plot(ep, rates[:, i], "o-", label=name)
ax[2].set(title="Mean firing rate (spikes/neuron/step)", xlabel="epoch"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 10. Final evaluation of best checkpoint

In [ ]:
model.load_state_dict(torch.load(CFG["ckpt"], map_location=device))
res = evaluate(model, test_dl, return_preds=True)
print(f"Test accuracy: {res['acc']:.2f}%   loss: {res['loss']:.4f}")
print("Firing rates:", dict(zip(SpikingLeNet5.LAYERS, np.round(res["rates"], 4))))
print()
print(classification_report(res["targets"], res["preds"], digits=4))

cm = confusion_matrix(res["targets"], res["preds"])
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set(xticks=range(10), yticks=range(10), xlabel="predicted", ylabel="true", title="Confusion matrix")
plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.show()

## 11. Output spike raster for sample digits

In [ ]:
x, y = next(iter(test_dl))
x, y = x[:4].to(device), y[:4]
with torch.no_grad():
    counts, _, out_train = model(rate_encode(x, CFG["T"], CFG["gain"]), record=True)   # (T,B,10)

fig, ax = plt.subplots(2, 4, figsize=(15, 5), gridspec_kw=dict(height_ratios=[1, 1.2]))
for k in range(4):
    ax[0, k].imshow(x[k, 0].cpu(), cmap="gray"); ax[0, k].axis("off")
    ax[0, k].set_title(f"true {y[k].item()} | pred {counts[k].argmax().item()}")
    tt, nn_ = np.nonzero(out_train[:, k].cpu().numpy())
    ax[1, k].scatter(tt, nn_, marker="|", s=80, c="k")
    ax[1, k].set(xlim=(-0.5, CFG["T"] - 0.5), ylim=(-0.5, 9.5), yticks=range(10),
                 xlabel="time step", ylabel="output neuron")
    ax[1, k].axhline(y[k].item(), color="tab:green", alpha=.25, lw=8)
plt.suptitle("Output-layer spike raster (green band = true class)")
plt.tight_layout(); plt.show()

## 12. Accuracy vs. inference time steps
The same trained weights evaluated with fewer/more time steps: the latency–accuracy trade-off.

In [ ]:
T_train = CFG["T"]
T_list  = [1, 2, 5, 10, 15, 20, 25, 30, 40]
accs = []
for T in T_list:
    CFG["T"] = T
    accs.append(evaluate(model, test_dl)["acc"])
    print(f"T={T:3d}  acc={accs[-1]:.2f}%")
CFG["T"] = T_train

plt.figure(figsize=(6, 3.8))
plt.plot(T_list, accs, "o-"); plt.axvline(T_train, ls="--", c="gray", label=f"T_train={T_train}")
plt.xlabel("inference time steps T"); plt.ylabel("test accuracy (%)"); plt.grid(alpha=.3); plt.legend()
plt.title("Accuracy vs. latency"); plt.tight_layout(); plt.show()